# <a id="intro-architecture"></a>
## 1. Introduction et Définition de l'Architecture de Base (LSTMTagger)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMTagger(nn.Module):
    """
    Modèle LSTM de base pour l'étiquetage morphosyntaxique (POS Tagging) séquence par séquence.
    """
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        # Couche d'Embedding : transforme les indices de mots en vecteurs denses
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # Couche LSTM : traite la séquence de vecteurs d'embeddings
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        # Couche Linéaire : projette l'état caché de l'espace LSTM vers l'espace des tags
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        # Passage dans la couche d'embedding -> forme (seq_len, embedding_dim)
        embeds = self.word_embeddings(sentence)
        
        # Redimensionnement pour le LSTM (seq_len, batch_size=1, embedding_dim)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        
        # Projection vers le nombre de classes (tags)
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        
        # Calcul de la distribution de probabilité via Log-Softmax
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

# <a id="bilstm-architecture"></a>
## 2. Architecture Bidirectionnelle (BiLSTM) avec Dropout et Batching

In [2]:
class AdvancedBiLSTMTagger(nn.Module):
    """
    Modèle BiLSTM avancé supportant :
    - La bidirectionnalité (contexte gauche et droit)
    - Le Dropout pour la régularisation
    - Les entrées par batch (batch_size > 1)
    """
    def __init__(self, vocab_size, tagset_size, embedding_dim=128, hidden_dim=256, dropout_p=0.3, padding_idx=0):
        super(AdvancedBiLSTMTagger, self).__init__()
        
        # Couche Embedding avec ignorage du token <PAD>
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, 
            embedding_dim=embedding_dim, 
            padding_idx=padding_idx
        )
        
        # Couche Dropout
        self.dropout = nn.Dropout(p=dropout_p)
        
        # LSTM Bidirectionnel (bidirectional=True)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_p
        )
        
        # En bidirectionnel, l'état caché retourné est de taille hidden_dim * 2
        self.fc = nn.Linear(hidden_dim * 2, tagset_size)

    def forward(self, x):
        # x : (batch_size, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch_size, seq_len, embedding_dim)
        
        lstm_out, _ = self.lstm(embedded)           # (batch_size, seq_len, hidden_dim * 2)
        
        logits = self.fc(self.dropout(lstm_out))     # (batch_size, seq_len, tagset_size)
        
        return logits

# <a id="test-instantiation"></a>
## 3. Test d'Instanciation et Validation de la Forme des Sorties (Dimensions Tenseurs)

In [3]:
# Hyperparamètres de test
VOCAB_SIZE = 100
TAGSET_SIZE = 15
EMBEDDING_DIM = 64
HIDDEN_DIM = 128

# Instanciation des modèles
simple_model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, VOCAB_SIZE, TAGSET_SIZE)
bilstm_model = AdvancedBiLSTMTagger(VOCAB_SIZE, TAGSET_SIZE, EMBEDDING_DIM, HIDDEN_DIM)

# Tenseur de test fictif (Batch de 2 phrases, longueur 5)
input_dummy = torch.randint(0, VOCAB_SIZE, (2, 5))

# Exécution du test sur le modèle BiLSTM
output_dummy = bilstm_model(input_dummy)

print("--- Validation des Dimensions ---")
print("Taille de l'entrée (Batch, Seq_Len)     :", input_dummy.shape)
print("Taille de la sortie (Batch, Seq_Len, Tags):", output_dummy.shape)

--- Validation des Dimensions ---
Taille de l'entrée (Batch, Seq_Len)     : torch.Size([2, 5])
Taille de la sortie (Batch, Seq_Len, Tags): torch.Size([2, 5, 15])
